# UI Element Detector — Phase 6 testing (Kaggle, GPU)

Runs the **test split** (13,208 images, never seen in training) through the trained models:

1. `run_detector_on_test.py` — YOLO26n predicts boxes on every test image.
2. `test_classifiers.py --source gt` — the 5 classifiers on ground-truth crops
   (primary cross-model comparison).
3. `test_classifiers.py --source det` — the 5 classifiers on the detector's own boxes,
   matched to ground truth by IoU >= 0.5 (end-to-end system quality).

Outputs go to `results/testing/` and are zipped at the end. **No training happens here.**

## Before you run

1. **Session options → Accelerator: GPU T4 x2**, **Internet: On**.
2. **Add Input → Your datasets → `ui-data`** (the data zip).
3. **Add Input → Your work → Notebook output** — pick the training notebook version whose
   output contains the trained checkpoints (`results/<model>/best.pt`,
   `yolo26n/weights/best.pt`). A `ui-results.zip` added as a dataset also works —
   the checkpoint cell unpacks it automatically.

Total runtime on a T4: roughly **45–75 minutes**.

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. Code (same clone-or-unzip fallback as the training notebook)

In [ ]:
import glob, os, pathlib, subprocess, zipfile

PROJECT = pathlib.Path('/kaggle/working/project')
REPO_URL = 'https://github.com/AnaraHayat/ui-Element-detector.git'

if not (PROJECT / 'scripts').is_dir():
    print('cloning', REPO_URL)
    r = subprocess.run(f'git clone -q {REPO_URL} {PROJECT}', shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('clone failed:', (r.stderr or '').strip()[:200])
        zips = (sorted(glob.glob('/kaggle/input/**/*code*.zip', recursive=True)) +
                sorted(glob.glob('/kaggle/working/*code*.zip')))
        assert zips, 'cannot get the code: enable Internet, or upload ui-element-detector-code.zip'
        print('unpacking code from', zips[0])
        zipfile.ZipFile(zips[0]).extractall(PROJECT)

os.chdir(PROJECT)
assert (PROJECT / 'scripts' / 'testing' / 'test_classifiers.py').exists(), (
    'the repo does not contain the Phase 6 testing scripts - the clone is stale')
!pip install -q timm ultralytics
print('code ready:', os.getcwd())

## 2. Data

Same auto-detection as the training notebook: an already-extracted dataset, or a
`*data*.zip` under `/kaggle/input` or `/kaggle/working`.

In [ ]:
import glob, os, pathlib, shutil, yaml

PROJECT = pathlib.Path('/kaggle/working/project')
YOLO = PROJECT / 'data' / 'yolo'
YOLO.mkdir(parents=True, exist_ok=True)
yaml_file = YOLO / 'data.yaml'


def has_images(root: pathlib.Path) -> bool:
    return (root / 'images' / 'train').is_dir() and any((root / 'images' / 'train').glob('*.jpg'))


def find_root() -> pathlib.Path | None:
    for pattern in ('/kaggle/input/**/data.yaml', '/kaggle/working/**/data.yaml'):
        for y in glob.glob(pattern, recursive=True):
            root = pathlib.Path(y).parent
            if has_images(root):
                return root
    return None


def find_zip() -> pathlib.Path | None:
    for pattern in ('/kaggle/input/**/*data*.zip', '/kaggle/working/*data*.zip'):
        for z in sorted(glob.glob(pattern, recursive=True)):
            p = pathlib.Path(z)
            if 'code' not in p.name.lower():
                return p
    return None


if yaml_file.exists():
    print('dataset already prepared:', YOLO)
else:
    root = find_root()
    if root is not None:
        print('using already-extracted dataset:', root)
        for sub in ('images', 'labels', 'class_index'):
            src, dst = root / sub, YOLO / sub
            if src.exists() and not dst.exists():
                try:
                    os.symlink(src, dst, target_is_directory=True)
                except OSError:
                    shutil.copytree(src, dst)
        cfg = yaml.safe_load((root / 'data.yaml').read_text())
        names = cfg['names']
        yaml_file.write_text(yaml.safe_dump({'train': 'images/train', 'val': 'images/val',
                                             'test': 'images/test', 'nc': len(names),
                                             'names': names}, sort_keys=False))
    else:
        zip_path = find_zip()
        assert zip_path is not None, 'no data found - add the ui-data dataset via Add Input'
        print('unpacking', zip_path, round(zip_path.stat().st_size / 1e9, 2), 'GB')
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(PROJECT / 'data')
        cfg = yaml.safe_load(yaml_file.read_text())
        cfg.pop('path', None)
        yaml_file.write_text(yaml.safe_dump(cfg, sort_keys=False))
        print('unpacked to', YOLO)

print('test images:', len(list((YOLO / 'images' / 'test').glob('*.jpg'))))
assert len(list((YOLO / 'images' / 'test').glob('*.jpg'))) > 12000, 'test split looks wrong'

## 3. Checkpoints

Finds `best.pt` for the 5 classifiers and `yolo26n/weights/best.pt` anywhere under
`/kaggle/input` (a training notebook's saved output, or a `ui-results*.zip` added as a
dataset, which is unpacked first) and copies them into `project/results/`.

In [ ]:
import glob, os, pathlib, shutil, zipfile

MODELS = ['convnext_tiny', 'convnextv2_tiny', 'fastvit_sa12', 'efficientvit_b1', 'repvit_m1_5']
PROJECT = pathlib.Path('/kaggle/working/project')

# a zip of the training results added as a dataset? unpack it first
for z in (glob.glob('/kaggle/input/**/*results*.zip', recursive=True) +
          glob.glob('/kaggle/working/*results*.zip')):
    if 'code' in pathlib.Path(z).name.lower():
        continue
    dest = pathlib.Path('/kaggle/working/unpacked_results')
    if not dest.exists():
        print('unpacking', z)
        zipfile.ZipFile(z).extractall(dest)

def find_one(rel: str):
    hits = (glob.glob(f'/kaggle/input/**/{rel}', recursive=True) +
            glob.glob(f'/kaggle/working/unpacked_results/**/{rel}', recursive=True) +
            glob.glob(f'/kaggle/working/**/{rel}', recursive=True))
    return pathlib.Path(hits[0]) if hits else None

LOGS = ['metrics.csv', 'history.json', 'val_per_class.json', 'config_used.yaml']
missing = []
for m in MODELS:
    dst = PROJECT / 'results' / m
    dst.mkdir(parents=True, exist_ok=True)
    ckpt = find_one(f'{m}/best.pt')
    if ckpt is None:
        missing.append(m)
        continue
    if not (dst / 'best.pt').exists():
        shutil.copy(ckpt, dst / 'best.pt')
    for log in LOGS:
        src = find_one(f'{m}/{log}')
        if src and not (dst / log).exists():
            shutil.copy(src, dst / log)

y_ckpt = find_one('yolo26n/weights/best.pt') or find_one('yolo26n/best.pt')
y_dir = PROJECT / 'results' / 'yolo26n' / 'weights'
y_dir.mkdir(parents=True, exist_ok=True)
if y_ckpt and not (y_dir / 'best.pt').exists():
    shutil.copy(y_ckpt, y_dir / 'best.pt')
for log in ('results.csv', 'args.yaml'):
    src = find_one(f'yolo26n/{log}')
    if src and not (PROJECT / 'results' / 'yolo26n' / log).exists():
        shutil.copy(src, PROJECT / 'results' / 'yolo26n' / log)

assert not missing, f'missing checkpoints for {missing} - add the training output via Add Input'
assert (y_dir / 'best.pt').exists(), 'yolo26n/weights/best.pt not found in the inputs'
for p in sorted((PROJECT / 'results').glob('*/best.pt')) + sorted((PROJECT / 'results').glob('*/weights/best.pt')):
    print(f'{str(p.relative_to(PROJECT)):<40}{p.stat().st_size / 1e6:7.1f} MB')

## 4. Verify all 6 checkpoints load (fast, catches corrupt files before hours of inference)

In [ ]:
import os, pathlib
PROJECT = pathlib.Path('/kaggle/working/project')
assert (PROJECT / 'scripts' / 'testing' / 'verify_checkpoints.py').exists(), \
    'project folder not found - run cells 1-3 (code, data, checkpoints) first'
os.chdir(PROJECT)
!python scripts/testing/verify_checkpoints.py
import torch
from ultralytics import YOLO
y = YOLO('results/yolo26n/weights/best.pt')
print('yolo26n loads OK,', sum(p.numel() for p in y.model.parameters()) / 1e6, 'M params')

## 5. Detector on the test split (~10-20 min)

In [ ]:
import os, pathlib
PROJECT = pathlib.Path('/kaggle/working/project')
assert (PROJECT / 'scripts' / 'testing' / 'run_detector_on_test.py').exists(), \
    'project folder not found - run cells 1-3 (code, data, checkpoints) first'
os.chdir(PROJECT)
!python scripts/testing/run_detector_on_test.py --device cuda
import csv
rows = list(csv.DictReader(open('results/testing/yolo26n/summary.csv')))
total = sum(int(r['boxes']) for r in rows)
print(f'{len(rows):,} images, {total:,} predicted boxes ({total / len(rows):.1f} per image)')

## 6. Classifiers on ground-truth crops (primary comparison, ~5-10 min each)

In [ ]:
import os
os.chdir('/kaggle/working/project')
MODELS = ['convnext_tiny', 'convnextv2_tiny', 'fastvit_sa12', 'efficientvit_b1', 'repvit_m1_5']
for m in MODELS:
    print('=' * 70); print('TESTING', m, '(ground-truth crops)')
    !python scripts/testing/test_classifiers.py --model {m} --source gt --num-workers 2 --device cuda

## 7. Classifiers on the detector's boxes (end-to-end, ~5 min each)

In [ ]:
import os
os.chdir('/kaggle/working/project')
MODELS = ['convnext_tiny', 'convnextv2_tiny', 'fastvit_sa12', 'efficientvit_b1', 'repvit_m1_5']
for m in MODELS:
    print('=' * 70); print('TESTING', m, '(detected crops)')
    !python scripts/testing/test_classifiers.py --model {m} --source det --num-workers 2 --device cuda
print()
for f in sorted(__import__('pathlib').Path('results/testing').glob('*_test.csv')):
    print(f.name, round(f.stat().st_size / 1e6, 1), 'MB')

## 8. Package

**Save Version** at the end (Quick Save is enough), then download `ui-testing.zip` from the
**Output** tab and send it back — Phase 7 (evaluation) runs on it.

In [ ]:
import os, shutil
os.chdir('/kaggle/working/project')
shutil.make_archive('/kaggle/working/ui-testing', 'zip', 'results/testing')
p = '/kaggle/working/ui-testing.zip'
print('saved:', p, round(os.path.getsize(p) / 1e6, 1), 'MB')